# M5 readiness — one notebook, multiple execution environments

**Purpose:** inspect installed SDKs and execute small, reproducible CPU or *explicitly authorised* GPU tests in the current Jupyter kernel. This is **not** a substitute for Slurm accounting, platform job submission or scientific validation.

Use separately provisioned kernels/immutable runners for CPU, NVIDIA GPU, and provider SDKs. Running this notebook in a Kubernetes workbench **does not** grant H200/A100 access. The GPU tests are disabled by default and must run inside an approved allocation exposing exactly the intended GPU(s).

No remote QPU is submitted from this notebook. The optional IBM check only discovers account-accessible backends; its API key and per-student instance CRN must be supplied as protected environment variables, never entered in notebook cells.

**Related working science:** [Bell CPU/GPU/QPU lesson](../01-intro-quantum-computing/02-bell-cpu-gpu-qpu.ipynb), [portfolio Ising CPU baseline](../05-quantum-finance/01-portfolio-ising-lightning-cpu.ipynb), and [M5 scientific acceptance roadmap](../../docs/M5-SCIENTIFIC-DELIVERY-ROADMAP.md).

In [ ]:
# Explicit opt-ins. GPU execution consumes facility resources; IBM below is discovery only.
ENABLE_GPU_SMOKES = False
ENABLE_IBM_DISCOVERY = False
WRITE_REPORT = False

from datetime import datetime, timezone
from importlib import metadata, util
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

checks = {}
def mark(name, status, detail):
    checks[name] = {"status": status, "detail": detail}
    print(f"{name:22s} {status:6s} {detail}")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Executable:", sys.executable)
print("UTC:", datetime.now(timezone.utc).isoformat())

# Device inventory is diagnostic, not an allocation or GPU isolation proof.
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,uuid", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=8, check=False
    )
    print("NVIDIA GPU inventory:", gpu.stdout.strip() if gpu.returncode == 0 else "unavailable")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("NVIDIA GPU inventory: unavailable in this kernel")

In [ ]:
distributions = {
    "Qiskit": "qiskit",
    "Qiskit Aer": "qiskit-aer",
    "Qiskit Aer GPU": "qiskit-aer-gpu",
    "PennyLane": "pennylane",
    "Lightning CPU": "pennylane-lightning",
    "Lightning GPU": "pennylane-lightning-gpu",
    "Cirq": "cirq-core",
    "NVIDIA cuQuantum": "cuquantum-python",
    "NVIDIA cuStateVec": "custatevec-cu12",
    "IQM Qiskit": "qiskit-iqm",
    "Pasqal Pulser": "pulser",
    "Pulser simulation": "pulser-simulation",
    "IBM Quantum Runtime": "qiskit-ibm-runtime",
}
installed = {}
for label, dist in distributions.items():
    try:
        installed[label] = metadata.version(dist)
    except metadata.PackageNotFoundError:
        installed[label] = None

for label, version in installed.items():
    print(f"{label:22s} {version or 'not installed in this kernel'}")
print("\nPackage presence alone is not proof of simulator execution or QPU entitlement.")

In [ ]:
# Real scientific smoke: |Phi+> = (|00>+|11>)/sqrt(2).
# 256 shots; outputs outside {00,11} are a numerical correctness failure.
SHOTS = 256
SEED = 7
try:
    from qiskit import QuantumCircuit
    from qiskit_aer import AerSimulator
except ImportError as exc:
    mark("Aer CPU Bell", "SKIP", f"Install Qiskit and Aer in this kernel: {exc}")
else:
    try:
        bell = QuantumCircuit(2)
        bell.h(0)
        bell.cx(0, 1)
        bell.measure_all()
        cpu = AerSimulator(method="statevector", device="CPU")
        cpu_counts = cpu.run(bell, shots=SHOTS, seed_simulator=SEED).result().get_counts()
        assert set(cpu_counts).issubset({"00", "11"})
        assert sum(cpu_counts.values()) == SHOTS
        mark("Aer CPU Bell", "PASS", f"Counts={cpu_counts}")
    except Exception as exc:
        mark("Aer CPU Bell", "FAIL", f"{type(exc).__name__}: {exc}")

# Reuse the existing quantum-workflows Lightning implementation.
if util.find_spec("pennylane") is None or util.find_spec("quantum_workflows") is None:
    mark("Lightning CPU Bell", "SKIP", "Requires PennyLane and quantum-workflows")
else:
    try:
        from quantum_workflows.lightning import sample_bell
        cpu_lightning_counts, backend = sample_bell(
            executor="lightning-cpu", shots=SHOTS, seed=SEED
        )
        assert backend == "lightning.qubit"
        assert set(cpu_lightning_counts).issubset({"00", "11"})
        assert sum(cpu_lightning_counts.values()) == SHOTS
        mark("Lightning CPU Bell", "PASS", f"Counts={cpu_lightning_counts}")
    except Exception as exc:
        mark("Lightning CPU Bell", "FAIL", f"{type(exc).__name__}: {exc}")

In [ ]:
if not ENABLE_GPU_SMOKES:
    mark("Aer GPU Bell", "TODO", "Enable only inside an approved GPU allocation")
    mark("Lightning GPU Bell", "TODO", "Enable only inside an approved GPU allocation")
else:
    # Allocation must expose only intended GPU(s). Do not benchmark a shared
    # GPU without an approved reservation / scheduler policy.
    try:
        from qiskit import QuantumCircuit
        from qiskit_aer import AerSimulator
        qc = QuantumCircuit(2)
        qc.h(0); qc.cx(0, 1); qc.measure_all()
        gpu_backend = AerSimulator(method="statevector", device="GPU")
        if "GPU" not in gpu_backend.available_devices():
            raise RuntimeError("This Aer installation does not expose GPU simulation")
        result = gpu_backend.run(qc, shots=SHOTS, seed_simulator=SEED).result()
        counts_gpu = result.get_counts()
        assert set(counts_gpu).issubset({"00", "11"})
        assert sum(counts_gpu.values()) == SHOTS
        mark("Aer GPU Bell", "PASS", f"Counts={counts_gpu}")
    except Exception as exc:
        mark("Aer GPU Bell", "FAIL", f"{type(exc).__name__}: {exc}")
    try:
        from quantum_workflows.lightning import sample_bell
        counts_lgpu, backend = sample_bell(
            executor="lightning-gpu", shots=SHOTS, seed=SEED
        )
        assert backend == "lightning.gpu"
        assert set(counts_lgpu).issubset({"00", "11"})
        assert sum(counts_lgpu.values()) == SHOTS
        mark("Lightning GPU Bell", "PASS", f"Counts={counts_lgpu}")
    except Exception as exc:
        mark("Lightning GPU Bell", "FAIL", f"{type(exc).__name__}: {exc}")

In [ ]:
if not ENABLE_IBM_DISCOVERY:
    mark("IBM classroom access", "TODO", "Account discovery is opt-in; no QPU job is submitted")
else:
    token = os.environ.get("IBM_QUANTUM_API_KEY")
    crn = os.environ.get("IBM_QUANTUM_INSTANCE_CRN")
    if not token or not crn:
        mark("IBM classroom access", "FAIL", "Missing private IBM API key or instance CRN")
    else:
        try:
            from qiskit_ibm_runtime import QiskitRuntimeService
            service = QiskitRuntimeService(token=token, instance=crn)
            names = [backend.name for backend in service.backends()]
            mark("IBM classroom access", "PASS", f"{len(names)} accessible backends (no jobs submitted)")
        except Exception as exc:
            mark("IBM classroom access", "FAIL", f"{type(exc).__name__}: {exc}")
    # Never print or save the token, CRN, or contents of environment variables.

print("\nSummary:", {key: val["status"] for key, val in checks.items()})
if WRITE_REPORT:
    path = Path("results/backend-readiness.json")
    path.parent.mkdir(parents=True, exist_ok=True)
    report = {
        "schema": "m5-backend-readiness/v1",
        "utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version.split()[0],
        "packages": installed,
        "checks": checks,
        "gpu_requested": ENABLE_GPU_SMOKES,
        "ibm_discovery_requested": ENABLE_IBM_DISCOVERY
    }
    path.write_text(json.dumps(report, indent=2) + "\n")
    print("Wrote redacted readiness report:", path)
    # A readiness report is not a scientific result/experiment manifest.